
# Right-boundary (GHZ initial state) \(\chi_2\) finite-size scaling

Identical analysis to `right_boundary_chi2_scaling_Julia_Px_eq_Pzz_fixed.ipynb`,
except the underlying simulation data comes from the GHZ-initial-state right-boundary
scan (`run_right_boundary_ghz_scan.jl` / `renyi2_right_boundary_ghz_core.jl`)
instead of the computational-basis-product-state initial condition.

This notebook reads `right_boundary_ghz_results_<timestamp>` (either a directory containing JSON files or one JSON file), fixes one value of \(P\), and compares

\[
\chi_2(L)=a+bL
\]

with

\[
\chi_2(L)=a+b\log L.
\]

## Why this test?

For the right-boundary order parameter

\[
Q=\sum_{i=1}^{L} O_i,
\]

suppose the stored quantity is

\[
M_2=\frac{\langle Q^2\rangle_2}{L^2},
\qquad
\langle A\rangle_2=
\frac{\operatorname{Tr}(A\rho^2)}{\operatorname{Tr}(\rho^2)}.
\]

The susceptibility-like quantity is then

\[
\chi_2
=
\frac{\langle Q^2\rangle_2}{L}
=
L M_2.
\]

Also,

\[
\chi_2
=
\frac{1}{L}\sum_{i,j}C_2(i,j)
\approx
\sum_{r=0}^{L-1}C_2(r).
\]

Therefore:

- short-range correlations: \(C_2(r)\sim e^{-r/\xi}\Rightarrow \chi_2\to\) constant;
- marginal/critical correlations: \(C_2(r)\sim 1/r\Rightarrow \chi_2\sim\log L\);
- long-range order: \(C_2(r)\to m_2^2\Rightarrow \chi_2\sim L\).

This is why plotting \(\chi_2\) against both \(L\) and \(\log L\) is physically useful.

> **Convention check:** this notebook defines `chi2 = L * M2_bar`. If Jake's \(X_2\) uses a different normalization, change the one marked line in the “Define observable” cell.



In [ ]:

# Install packages only when they are missing.
import Pkg

for pkg in ["JSON3", "DataFrames", "Plots"]
    if Base.find_package(pkg) === nothing
        Pkg.add(pkg)
    end
end

using JSON3
using DataFrames
using Statistics
using LinearAlgebra
using Printf
using Plots

default(
    framestyle = :box,
    grid = false,
    markerstrokewidth = 0.8,
    linewidth = 2,
    size = (760, 500),
)



## 1. Configuration

Place this notebook beside the results directory/file, or replace `DATA_PATH` with an absolute path.

Examples:

```julia
DATA_PATH = "right_boundary_ghz_results_20260710_1229"
DATA_PATH = "right_boundary_ghz_results_20260710_1229.json"
DATA_PATH = "/Users/qia/.../right_boundary_ghz_results_20260710_1229"
```


In [ ]:

# ---------- User settings ----------

DATA_PATH = "right_boundary_ghz_results_20260710_1229"  # collected from collect_right_boundary_ghz_results.sh

# The probability to hold fixed.
P_FIELD  = :P_x
P_TARGET = 0.30
P_TOL    = 1e-10

# :exact   -> stop if P_TARGET is unavailable
# :nearest -> automatically use the closest available P
P_SELECTION_MODE = :nearest

# These filters are applied only when the corresponding columns exist.
FIXED_FILTERS = Dict(
    :lambda_x  => 0.7,
    :lambda_zz => 0.0,
)

# System sizes expected in this dataset.
EXPECTED_L = [8, 16, 24, 32]

# Directory used when saving figures/tables.
OUTPUT_DIR = "right_boundary_ghz_chi2_scaling_output"


# The right-boundary scan in the figure uses P_x = P_zz.
# Keep only rows satisfying that relation.
COUPLE_PX_PZZ = true

# Require enough system sizes for a fit.
MIN_DISTINCT_L = 3


## 2. Load all JSON records

In [ ]:

function resolve_data_path(path::AbstractString)
    if ispath(path)
        return path
    elseif isfile(path * ".json")
        return path * ".json"
    else
        error(
            "Could not find '$path' or '$path.json'. " *
            "Set DATA_PATH to the correct directory or JSON file."
        )
    end
end

function json_files_under(path::AbstractString)
    resolved = resolve_data_path(path)

    if isfile(resolved)
        return [resolved]
    end

    files = String[]
    for (root, _, names) in walkdir(resolved)
        for name in names
            if endswith(lowercase(name), ".json")
                push!(files, joinpath(root, name))
            end
        end
    end

    sort!(files)
    isempty(files) && error("No .json files found under '$resolved'.")
    return files
end

function object_to_dict(obj)
    return Dict{String,Any}(string(k) => v for (k, v) in pairs(obj))
end

function normalize_json_object(obj, path::AbstractString)
    # Top-level JSON array: [ {...}, {...} ]
    if obj isa AbstractVector
        return [object_to_dict(item) for item in obj]
    end

    # Top-level JSON object: {...}
    if obj isa JSON3.Object
        # Also support {"results": [ ... ]}.
        if haskey(obj, :results) && obj[:results] isa AbstractVector
            return [object_to_dict(item) for item in obj[:results]]
        elseif haskey(obj, "results") && obj["results"] isa AbstractVector
            return [object_to_dict(item) for item in obj["results"]]
        else
            return [object_to_dict(obj)]
        end
    end

    error("Unsupported top-level JSON structure in '$path'.")
end

function try_parse_json_lines(text::AbstractString, path::AbstractString)
    rows = Dict{String,Any}[]

    for (line_number, line) in enumerate(eachline(IOBuffer(text)))
        stripped = strip(line)
        isempty(stripped) && continue

        try
            obj = JSON3.read(stripped)
            append!(rows, normalize_json_object(obj, path))
        catch err
            @warn(
                "JSON-lines fallback failed; this file will be skipped.",
                file = path,
                line = line_number,
                error = sprint(showerror, err),
            )
            return Dict{String,Any}[]
        end
    end

    return rows
end

function records_from_json_file(path::AbstractString)
    text = read(path, String)

    # This directly fixes:
    # "invalid JSON at byte position 0 ... UnexpectedEOF"
    # which normally means an empty or whitespace-only file.
    if isempty(strip(text))
        @warn "Skipping empty JSON file." file=path
        return Dict{String,Any}[]
    end

    try
        obj = JSON3.read(text)
        return normalize_json_object(obj, path)
    catch err
        # Some result collectors write one JSON object per line rather than
        # one top-level JSON array. Try that format before skipping the file.
        rows = try_parse_json_lines(text, path)

        if !isempty(rows)
            @info "Parsed file as JSON Lines." file=path records=length(rows)
            return rows
        end

        @warn(
            "Skipping malformed or incomplete JSON file.",
            file = path,
            bytes = filesize(path),
            error = sprint(showerror, err),
        )
        return Dict{String,Any}[]
    end
end

function dicts_to_dataframe(rows::Vector{Dict{String,Any}})
    isempty(rows) && return DataFrame()

    all_keys = sort(unique(vcat([collect(keys(row)) for row in rows]...)))
    columns = Dict{Symbol,Vector{Any}}()

    for key in all_keys
        columns[Symbol(key)] = [get(row, key, missing) for row in rows]
    end

    return DataFrame(columns)
end

json_files = json_files_under(DATA_PATH)
println("Found $(length(json_files)) JSON file(s).")

# Print zero-byte files before parsing so they are easy to locate.
empty_files = filter(file -> filesize(file) == 0, json_files)
if !isempty(empty_files)
    println("\nZero-byte JSON files that will be skipped:")
    foreach(file -> println("  ", file), empty_files)
end

records = Dict{String,Any}[]
files_with_records = 0

for file in json_files
    file_records = records_from_json_file(file)
    if !isempty(file_records)
        append!(records, file_records)
        files_with_records += 1
    end
end

isempty(records) && error(
    "No valid JSON records were loaded. Check the warning messages above " *
    "and verify that DATA_PATH points to the completed result files."
)

df_raw = dicts_to_dataframe(records)

println("\nLoaded $(nrow(df_raw)) record(s) from $files_with_records valid file(s).")
println("Columns:")
println(names(df_raw))


## 3. Clean numeric columns and inspect available parameters

In [ ]:

function convert_numeric_column!(df::DataFrame, col::Symbol; integer::Bool=false)
    col ∉ propertynames(df) && return df

    if integer
        df[!, col] = [
            ismissing(x) ? missing : Int(round(Float64(x)))
            for x in df[!, col]
        ]
    else
        df[!, col] = [
            ismissing(x) ? missing : Float64(x)
            for x in df[!, col]
        ]
    end

    return df
end

df = copy(df_raw)

convert_numeric_column!(df, :L; integer=true)

for col in [
    :P_x, :P_zz, :lambda_x, :lambda_zz,
    :M2_bar, :M4_bar, :B, :purity_bar,
    :n_valid, :ntrials
]
    convert_numeric_column!(df, col)
end

required_columns = [:L, :M2_bar, P_FIELD]
missing_columns = [c for c in required_columns if c ∉ propertynames(df)]
isempty(missing_columns) || error("Missing required columns: $missing_columns")

available_L = sort(unique(collect(skipmissing(df.L))))
available_P = sort(unique(collect(skipmissing(df[!, P_FIELD]))))

println("Available L values: ", available_L)
println("Available $(P_FIELD) values: ", available_P)

missing_expected_L = setdiff(EXPECTED_L, available_L)
if !isempty(missing_expected_L)
    @warn "Expected system sizes are missing from the full dataset" missing_expected_L
end



## 4. Define \(X_2\) / \(\chi_2\)

The supplied JSON stores `M2_bar`. Based on the normalization

\[
M_2=\frac{\langle Q^2\rangle_2}{L^2},
\]

the susceptibility is

\[
\chi_2=L M_2.
\]

The code also stores the unnormalized second moment

\[
\langle Q^2\rangle_2=L^2M_2
\]

for reference.


In [ ]:

# ---------- Change this line only if Jake uses another normalization ----------
df.chi2 = df.L .* df.M2_bar

# Useful reference quantity:
df.Q2_unnormalized = (df.L .^ 2) .* df.M2_bar

first(df[:, [:L, P_FIELD, :M2_bar, :chi2, :Q2_unnormalized]], 10)


## 5. Apply fixed parameters and select one \(P\)

In [ ]:

function approximately_equal(x, target; atol=1e-10)
    return !ismissing(x) && isapprox(Float64(x), Float64(target); atol=atol, rtol=0)
end

df_fixed = copy(df)

println("Rows before fixed-parameter filters: ", nrow(df_fixed))

# Apply only parameters that are truly fixed during the scan.
for (col, target) in FIXED_FILTERS
    if col in propertynames(df_fixed)
        before = nrow(df_fixed)
        df_fixed = filter(
            row -> approximately_equal(row[col], target; atol=P_TOL),
            df_fixed,
        )
        println("Filter $col = $target: $before -> $(nrow(df_fixed)) rows")
    else
        @warn "Filter column is absent; skipping this filter." column=col
    end
end

# The plotted scan says P_x = P_zz, so enforce equality rather than P_zz = 0.
if COUPLE_PX_PZZ && (:P_x in propertynames(df_fixed)) && (:P_zz in propertynames(df_fixed))
    before = nrow(df_fixed)
    df_fixed = filter(
        row -> !ismissing(row.P_x) &&
               !ismissing(row.P_zz) &&
               isapprox(Float64(row.P_x), Float64(row.P_zz); atol=P_TOL, rtol=0),
        df_fixed,
    )
    println("Filter P_x = P_zz: $before -> $(nrow(df_fixed)) rows")
end

isempty(df_fixed) && error(
    "No rows remain after the fixed-parameter filters. " *
    "Check lambda_x, lambda_zz, and the P_x=P_zz condition."
)

# Summarize every available P by BOTH record count and number of distinct L.
P_counts = combine(
    groupby(dropmissing(df_fixed, P_FIELD), P_FIELD),
    nrow => :n_records,
    :L => (x -> length(unique(x))) => :n_distinct_L,
    :L => (x -> join(sort(unique(x)), ", ")) => :available_L,
)
sort!(P_counts, P_FIELD)

println("\nAvailable $(P_FIELD) values after filtering:")
display(P_counts)

# Only P values with enough distinct sizes are eligible for fitting.
eligible = filter(row -> row.n_distinct_L >= MIN_DISTINCT_L, P_counts)

isempty(eligible) && error(
    "No P value has at least $MIN_DISTINCT_L distinct system sizes. " *
    "Inspect P_counts above to see which L values are present."
)

available_P_eligible = Float64.(eligible[!, P_FIELD])
requested_P = Float64(P_TARGET)

exact_index = findfirst(
    p -> isapprox(p, requested_P; atol=P_TOL, rtol=0),
    available_P_eligible,
)

if exact_index !== nothing
    P_USED = available_P_eligible[exact_index]
elseif P_SELECTION_MODE == :nearest
    nearest_index = argmin(abs.(available_P_eligible .- requested_P))
    P_USED = available_P_eligible[nearest_index]
    @warn(
        "Requested P is unavailable with enough L values; using nearest eligible P.",
        requested = requested_P,
        used = P_USED,
    )
elseif P_SELECTION_MODE == :exact
    nearest = available_P_eligible[argmin(abs.(available_P_eligible .- requested_P))]
    error(
        "P_TARGET=$requested_P is unavailable with at least $MIN_DISTINCT_L sizes. " *
        "Nearest eligible value is $nearest."
    )
else
    error("P_SELECTION_MODE must be :exact or :nearest.")
end

df_P = filter(
    row -> approximately_equal(row[P_FIELD], P_USED; atol=P_TOL),
    df_fixed,
)

# If several records exist for the same L, average them.
safe_std(v) = length(v) > 1 ? std(v) : 0.0

summary_P = combine(
    groupby(df_P, :L),
    :chi2 => mean => :chi2_mean,
    :chi2 => safe_std => :chi2_std_across_records,
    :M2_bar => mean => :M2_mean,
    nrow => :n_records,
)

sort!(summary_P, :L)

println("\nRequested $(P_FIELD) = $P_TARGET")
println("Actually using $(P_FIELD) = $P_USED")
println("Rows at selected P: ", nrow(df_P))
println("Distinct L at selected P: ", sort(unique(df_P.L)))
println("Summary table:")
display(summary_P)

if length(unique(summary_P.L)) < MIN_DISTINCT_L
    error(
        "Selected P=$P_USED has only $(length(unique(summary_P.L))) distinct L values. " *
        "This should not happen because P was selected from the eligible table."
    )
end

missing_L_here = setdiff(EXPECTED_L, collect(summary_P.L))
if !isempty(missing_L_here)
    @warn "Some expected system sizes are missing at the selected P." missing_L_here
end

summary_P



### Important: this scan uses \(P_x=P_{zz}\)

The Binder plot has horizontal axis \(P_x=P_{zz}\). Therefore `P_zz` must **not** be fixed to zero while scanning \(P\).

The notebook now:

1. fixes only \(\lambda_x=0.7\) and \(\lambda_{zz}=0\);
2. keeps rows satisfying \(P_x=P_{zz}\);
3. counts the distinct \(L\) values at every \(P\);
4. chooses only a \(P\) with at least three distinct system sizes.

The full Binder figure having four curves means the whole dataset contains four \(L\) values. The fit additionally requires that the **same fixed \(P\)** contains at least three of those sizes after all filters.



## 6. Fit \(a+bL\) and \(a+b\log L\)

With only four sizes, these fits are diagnostic rather than a final finite-size-scaling result. The most useful checks are:

1. which model has smaller residuals;
2. whether the conclusion changes after dropping \(L=8\);
3. whether the trend is physically consistent across nearby \(P\) values.


In [ ]:

function affine_fit(x::AbstractVector, y::AbstractVector)
    x = Float64.(x)
    y = Float64.(y)

    X = hcat(ones(length(x)), x)
    beta = X \ y
    yhat = X * beta
    residuals = y .- yhat

    rss = sum(abs2, residuals)
    tss = sum(abs2, y .- mean(y))
    r2 = tss > 0 ? 1 - rss / tss : NaN
    rmse = sqrt(rss / length(y))

    return (
        intercept = beta[1],
        slope = beta[2],
        yhat = yhat,
        residuals = residuals,
        rss = rss,
        rmse = rmse,
        r2 = r2,
    )
end

Lvals = Float64.(summary_P.L)
yvals = Float64.(summary_P.chi2_mean)

length(unique(Lvals)) >= 3 || error("Need at least three distinct L values.")

fit_linear_L = affine_fit(Lvals, yvals)
fit_linear_logL = affine_fit(log.(Lvals), yvals)

comparison = DataFrame(
    model = ["chi2 = a + b L", "chi2 = a + b log(L)"],
    intercept = [fit_linear_L.intercept, fit_linear_logL.intercept],
    slope = [fit_linear_L.slope, fit_linear_logL.slope],
    RSS = [fit_linear_L.rss, fit_linear_logL.rss],
    RMSE = [fit_linear_L.rmse, fit_linear_logL.rmse],
    R2 = [fit_linear_L.r2, fit_linear_logL.r2],
)

comparison


In [ ]:

@printf(
    "Linear in L:      chi2 = %.6g + %.6g L,       R² = %.6f, RSS = %.6g\n",
    fit_linear_L.intercept,
    fit_linear_L.slope,
    fit_linear_L.r2,
    fit_linear_L.rss,
)

@printf(
    "Linear in log L:  chi2 = %.6g + %.6g log L,   R² = %.6f, RSS = %.6g\n",
    fit_linear_logL.intercept,
    fit_linear_logL.slope,
    fit_linear_logL.r2,
    fit_linear_logL.rss,
)

if fit_linear_logL.rss < fit_linear_L.rss
    println("For these four sizes, the log(L) model has the smaller in-sample RSS.")
else
    println("For these four sizes, the linear-in-L model has the smaller in-sample RSS.")
end


## 7. Plot the same data in both coordinate choices

In [ ]:

Lgrid = range(minimum(Lvals), maximum(Lvals), length=300)

linear_curve = fit_linear_L.intercept .+ fit_linear_L.slope .* Lgrid
log_curve = fit_linear_logL.intercept .+ fit_linear_logL.slope .* log.(Lgrid)

yerr = summary_P.chi2_std_across_records

p_linear_axis = scatter(
    Lvals,
    yvals;
    yerror = yerr,
    xlabel = "L",
    ylabel = "χ₂ = L M₂",
    title = "Fixed $(P_FIELD) = $P_USED: ordinary L axis (GHZ initial state)",
    label = "data",
    markersize = 6,
)

plot!(
    p_linear_axis,
    Lgrid,
    linear_curve;
    label = @sprintf("a + bL  (R²=%.4f)", fit_linear_L.r2),
)

plot!(
    p_linear_axis,
    Lgrid,
    log_curve;
    linestyle = :dash,
    label = @sprintf("a + b log L  (R²=%.4f)", fit_linear_logL.r2),
)

p_linear_axis


In [ ]:

p_log_axis = scatter(
    Lvals,
    yvals;
    yerror = yerr,
    xscale = :log10,
    xlabel = "L  (log scale)",
    ylabel = "χ₂ = L M₂",
    title = "Fixed $(P_FIELD) = $P_USED: logarithmic x-axis (GHZ initial state)",
    label = "data",
    markersize = 6,
    xticks = (Lvals, string.(Int.(Lvals))),
)

# A + B log(L) appears as a straight line on this semilog-x plot.
plot!(
    p_log_axis,
    Lgrid,
    log_curve;
    label = @sprintf("a + b log L  (R²=%.4f)", fit_linear_logL.r2),
)

plot!(
    p_log_axis,
    Lgrid,
    linear_curve;
    linestyle = :dash,
    label = @sprintf("a + bL  (R²=%.4f)", fit_linear_L.r2),
)

p_log_axis



## 8. Direct slope check

For linear growth,

\[
\frac{\Delta\chi_2}{\Delta L}
\]

should be approximately constant.

For logarithmic growth,

\[
\frac{\Delta\chi_2}{\Delta\log L}
\]

should be approximately constant.


In [ ]:

slope_table = DataFrame(
    L_left = Lvals[1:end-1],
    L_right = Lvals[2:end],
    delta_chi2 = diff(yvals),
    slope_vs_L = diff(yvals) ./ diff(Lvals),
    slope_vs_logL = diff(yvals) ./ diff(log.(Lvals)),
)

slope_table



## 9. Repeat after dropping the smallest size

A fit that changes dramatically after removing \(L=8\) is probably dominated by finite-size corrections.


In [ ]:

if length(Lvals) >= 4
    keep = 2:length(Lvals)

    fit_L_no_small = affine_fit(Lvals[keep], yvals[keep])
    fit_log_no_small = affine_fit(log.(Lvals[keep]), yvals[keep])

    stability_table = DataFrame(
        model = [
            "a + bL, all L",
            "a + bL, drop smallest L",
            "a + b logL, all L",
            "a + b logL, drop smallest L",
        ],
        slope = [
            fit_linear_L.slope,
            fit_L_no_small.slope,
            fit_linear_logL.slope,
            fit_log_no_small.slope,
        ],
        RSS = [
            fit_linear_L.rss,
            fit_L_no_small.rss,
            fit_linear_logL.rss,
            fit_log_no_small.rss,
        ],
        R2 = [
            fit_linear_L.r2,
            fit_L_no_small.r2,
            fit_linear_logL.r2,
            fit_log_no_small.r2,
        ],
    )

    stability_table
else
    println("Fewer than four sizes; skipping the drop-smallest-size comparison.")
end


## 10. Compare the two scaling forms for every available \(P\)

In [ ]:

function summarize_scaling_by_P(df::DataFrame, pfield::Symbol)
    output = NamedTuple[]

    for p in sort(unique(collect(skipmissing(df[!, pfield]))))
        sub = filter(row -> approximately_equal(row[pfield], p; atol=P_TOL), df)

        grouped = combine(
            groupby(sub, :L),
            :chi2 => mean => :chi2_mean,
        )
        sort!(grouped, :L)

        if nrow(grouped) < 3
            continue
        end

        L = Float64.(grouped.L)
        y = Float64.(grouped.chi2_mean)

        fit_L = affine_fit(L, y)
        fit_logL = affine_fit(log.(L), y)

        preferred = fit_logL.rss < fit_L.rss ? "log(L)" : "L"

        push!(
            output,
            (
                P = Float64(p),
                n_sizes = length(L),
                slope_L = fit_L.slope,
                R2_L = fit_L.r2,
                RSS_L = fit_L.rss,
                slope_logL = fit_logL.slope,
                R2_logL = fit_logL.r2,
                RSS_logL = fit_logL.rss,
                preferred_by_RSS = preferred,
                RSS_log_minus_linear = fit_logL.rss - fit_L.rss,
            ),
        )
    end

    return DataFrame(output)
end

all_P_summary = summarize_scaling_by_P(df_fixed, P_FIELD)
all_P_summary



## 11. Save outputs

The notebook saves both figures and the model-comparison tables.


In [ ]:

mkpath(OUTPUT_DIR)

savefig(
    p_linear_axis,
    joinpath(
        OUTPUT_DIR,
        "chi2_vs_L_$(P_FIELD)_$(P_USED).png",
    ),
)

savefig(
    p_log_axis,
    joinpath(
        OUTPUT_DIR,
        "chi2_vs_logL_$(P_FIELD)_$(P_USED).png",
    ),
)

using DelimitedFiles

function write_dataframe_csv(path::AbstractString, table::DataFrame)
    open(path, "w") do io
        println(io, join(string.(names(table)), ","))
        for row in eachrow(table)
            println(io, join(string.(collect(row)), ","))
        end
    end
end

write_dataframe_csv(
    joinpath(OUTPUT_DIR, "fixed_P_model_comparison.csv"),
    comparison,
)

write_dataframe_csv(
    joinpath(OUTPUT_DIR, "all_P_scaling_summary.csv"),
    all_P_summary,
)

println("Saved outputs to: ", abspath(OUTPUT_DIR))



## How to interpret the result

For a fixed \(P\):

- \(\chi_2\) approximately constant with \(L\): short-range/disordered behavior;
- \(\chi_2\) linear in \(\log L\): consistent with \(C_2(r)\sim 1/r\);
- \(\chi_2\) linear in \(L\): consistent with long-range order;
- neither fit is stable: likely finite-size corrections or another power law.

With only \(L=8,16,24,32\), do not identify the asymptotic scaling from \(R^2\) alone. Check nearby \(P\), compare residuals, and test whether dropping \(L=8\) changes the conclusion.
